# Hybrid Quantum-Classical Rebalancing (Business Notebook)

This notebook is a business-friendly version of the PoC script.

It demonstrates:
- simulated market regimes
- quantum-assisted portfolio weight generation
- classical optimization with risk and transaction cost controls
- KPI summary for decision support

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Tuple

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

: 

## 1) Configuration
Adjust these parameters to test different scenarios.

In [ ]:
@dataclass
class PoCConfig:
    n_assets: int = 3
    n_steps: int = 180
    lookback: int = 20
    random_search_samples: int = 80
    risk_aversion: float = 8.0
    transaction_cost: float = 0.002
    seed: int = 7

## 2) Core Functions
These are the same algorithms as in the script version.

In [ ]:
def quantum_policy(features: np.ndarray, theta: np.ndarray) -> np.ndarray:
    n_assets = len(features)
    qc = QuantumCircuit(n_assets)

    for qubit, value in enumerate(features):
        qc.ry(float(value), qubit)

    for qubit in range(n_assets - 1):
        qc.cx(qubit, qubit + 1)

    for qubit in range(n_assets):
        qc.ry(float(theta[qubit]), qubit)
        qc.rz(float(theta[n_assets + qubit]), qubit)

    for qubit in range(n_assets - 1, 0, -1):
        qc.cx(qubit, qubit - 1)

    for qubit in range(n_assets):
        qc.ry(float(theta[2 * n_assets + qubit]), qubit)

    probabilities = Statevector.from_instruction(qc).probabilities()
    p_one = np.zeros(n_assets, dtype=float)
    for basis, prob in enumerate(probabilities):
        for asset in range(n_assets):
            if (basis >> asset) & 1:
                p_one[asset] += prob

    raw_scores = p_one + 1e-9
    return raw_scores / raw_scores.sum()


def simulate_market(n_steps: int, n_assets: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    returns = np.zeros((n_steps, n_assets))

    drift_regimes = np.array([
        [0.0012, 0.0008, 0.0010],
        [-0.0003, 0.0014, 0.0006],
        [0.0016, -0.0002, 0.0009],
    ])[:, :n_assets]

    base_cov = np.array([
        [0.00030, 0.00009, 0.00006],
        [0.00009, 0.00024, 0.00008],
        [0.00006, 0.00008, 0.00028],
    ])
    covariance = base_cov[:n_assets, :n_assets]

    regime = 0
    for t in range(n_steps):
        if t > 0 and t % 45 == 0:
            regime = (regime + 1) % drift_regimes.shape[0]
        returns[t] = rng.multivariate_normal(mean=drift_regimes[regime], cov=covariance)

    return returns


def build_features(window_returns: np.ndarray) -> np.ndarray:
    mean_signal = window_returns.mean(axis=0)
    volatility = window_returns.std(axis=0) + 1e-9
    momentum = window_returns[-5:].mean(axis=0)
    zscore = (momentum - mean_signal) / volatility
    scaled = np.tanh(3.5 * zscore)
    return np.pi * scaled


def evaluate_candidate(
    theta: np.ndarray,
    features: np.ndarray,
    mu: np.ndarray,
    cov: np.ndarray,
    prev_weights: np.ndarray,
    risk_aversion: float,
    transaction_cost: float,
) -> Tuple[float, np.ndarray]:
    candidate_weights = quantum_policy(features=features, theta=theta)
    expected_return = float(candidate_weights @ mu)
    variance = float(candidate_weights @ cov @ candidate_weights)
    turnover = float(np.abs(candidate_weights - prev_weights).sum())
    utility = expected_return - risk_aversion * variance - transaction_cost * turnover
    return utility, candidate_weights


def optimize_quantum_parameters(
    rng: np.random.Generator,
    features: np.ndarray,
    mu: np.ndarray,
    cov: np.ndarray,
    prev_weights: np.ndarray,
    theta_center: np.ndarray,
    random_search_samples: int,
    risk_aversion: float,
    transaction_cost: float,
) -> Tuple[np.ndarray, np.ndarray, float]:
    best_theta = theta_center.copy()
    best_utility, best_weights = evaluate_candidate(
        theta=best_theta,
        features=features,
        mu=mu,
        cov=cov,
        prev_weights=prev_weights,
        risk_aversion=risk_aversion,
        transaction_cost=transaction_cost,
    )

    for _ in range(random_search_samples):
        candidate_theta = theta_center + rng.normal(0.0, 0.35, size=theta_center.shape[0])
        utility, candidate_weights = evaluate_candidate(
            theta=candidate_theta,
            features=features,
            mu=mu,
            cov=cov,
            prev_weights=prev_weights,
            risk_aversion=risk_aversion,
            transaction_cost=transaction_cost,
        )
        if utility > best_utility:
            best_utility = utility
            best_theta = candidate_theta
            best_weights = candidate_weights

    return best_theta, best_weights, best_utility

## 3) Simulation Runner
Returns both detailed logs and top-level KPIs.

In [ ]:
def run_hybrid_rebalancing(config: PoCConfig):
    rng = np.random.default_rng(config.seed)
    returns = simulate_market(
        n_steps=config.n_steps,
        n_assets=config.n_assets,
        seed=config.seed,
    )

    theta = rng.normal(0.0, 0.5, size=3 * config.n_assets)
    weights = np.full(config.n_assets, 1.0 / config.n_assets)

    portfolio_returns = []
    wealth = [1.0]
    turnover_log = []
    utility_log = []
    weight_log = []

    for t in range(config.lookback, config.n_steps):
        window = returns[t - config.lookback : t]
        features = build_features(window)
        mu = window.mean(axis=0)
        cov = np.cov(window, rowvar=False)

        theta, new_weights, utility = optimize_quantum_parameters(
            rng=rng,
            features=features,
            mu=mu,
            cov=cov,
            prev_weights=weights,
            theta_center=theta,
            random_search_samples=config.random_search_samples,
            risk_aversion=config.risk_aversion,
            transaction_cost=config.transaction_cost,
        )

        turnover = float(np.abs(new_weights - weights).sum())
        realized = float(new_weights @ returns[t] - config.transaction_cost * turnover)

        weights = new_weights
        portfolio_returns.append(realized)
        wealth.append(wealth[-1] * (1.0 + realized))
        turnover_log.append(turnover)
        utility_log.append(utility)
        weight_log.append(weights.copy())

    portfolio_returns_arr = np.array(portfolio_returns)
    weight_log_arr = np.array(weight_log)

    ann_factor = 252
    avg_daily = float(portfolio_returns_arr.mean())
    vol_daily = float(portfolio_returns_arr.std() + 1e-12)
    sharpe = float(np.sqrt(ann_factor) * avg_daily / vol_daily)
    cumulative_return = float(wealth[-1] - 1.0)

    summary = {
        "assets": config.n_assets,
        "steps": config.n_steps,
        "lookback": config.lookback,
        "final_wealth": float(wealth[-1]),
        "cumulative_return_pct": 100.0 * cumulative_return,
        "avg_daily_return_pct": 100.0 * avg_daily,
        "daily_volatility_pct": 100.0 * vol_daily,
        "annualized_sharpe": sharpe,
        "avg_turnover": float(np.mean(turnover_log)),
        "avg_utility": float(np.mean(utility_log)),
    }

    return summary, wealth, weight_log_arr

## 4) Run and View KPI Output
Execute this cell to run the full workflow.

In [ ]:
config = PoCConfig()
summary, wealth, weight_log = run_hybrid_rebalancing(config)

print("Hybrid quantum-classical dynamic rebalancing PoC")
print("=" * 56)
print(f"Assets: {summary['assets']}, Steps: {summary['steps']}, Lookback: {summary['lookback']}")
print(f"Final wealth: {summary['final_wealth']:.4f}")
print(f"Cumulative return: {summary['cumulative_return_pct']:.2f}%")
print(f"Average daily return: {summary['avg_daily_return_pct']:.3f}%")
print(f"Daily volatility: {summary['daily_volatility_pct']:.3f}%")
print(f"Sharpe (annualized): {summary['annualized_sharpe']:.2f}")
print(f"Average turnover per rebalance: {summary['avg_turnover']:.3f}")
print(f"Average utility score: {summary['avg_utility']:.6f}")

## 5) Last 5 Rebalances
Shows the final portfolio allocations generated by the model.

In [ ]:
n_last = 5
start_idx = max(0, len(weight_log) - n_last)

print("Last 5 rebalances (weights):")
for i, row in enumerate(weight_log[-n_last:], start=start_idx + 1):
    weight_text = ", ".join(f"{w:.3f}" for w in row)
    print(f"t={i:3d} -> [{weight_text}]")